In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1. Import Data

In [ ]:
import torch

print('CUDA available :', torch.cuda.is_available())
print('GPU name       :', torch.cuda.get_device_name(0))
print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [1]:
#INPUT_FILE = "/content/drive/MyDrive/FYR_UOK/Final yr research UOK/OBJ-1/Global_data_process/Subsets/Gold_dataset.csv"
INPUT_FILE = "/Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/14_final_hybrid approach/Gold_dataset.csv"

In [2]:
import pandas as pd
gold_df_input = pd.read_csv(INPUT_FILE)
gold_df_input.head()

/Users/subhashbandaraekanayake/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Title,Date,Platform,Country,Description,text
0,CGTN Europe [cgtneurope],2024-09-20,TikTok,China,Brave panda conquers raging river in Shaanxi! ...,victory witness action rivercrossingchamp
1,CGTN Europe [cgtneurope],2024-09-20,TikTok,China,Israeli Air Force planes launched an airstrike...,force headquarters security health least three...
2,CGTN Europe [cgtneurope],2024-09-21,TikTok,China,The far-right Alternative for Germany (AfD) pa...,right election election
3,CGTN Europe [cgtneurope],2024-09-20,TikTok,China,Chechen leader Ramzan Kadyrov shared a video s...,temporarily territory vehicle front teslacyber...
4,CGTN Europe [cgtneurope],2024-09-18,TikTok,China,An explosion at a Russian military depot in th...,follow authority fire interception town torope...


In [3]:
df_input = gold_df_input.drop(columns=["Title", "Date","text"])

In [4]:
df_input.shape

(425000, 3)

In [5]:
df_input.head()

,Platform,Country,Description
0,TikTok,China,Brave panda conquers raging river in Shaanxi! ...
1,TikTok,China,Israeli Air Force planes launched an airstrike...
2,TikTok,China,The far-right Alternative for Germany (AfD) pa...
3,TikTok,China,Chechen leader Ramzan Kadyrov shared a video s...
4,TikTok,China,An explosion at a Russian military depot in th...


In [6]:
df_input['Platform'].unique()

array(['TikTok', 'Facebook', 'YouTube', 'Telegram', 'Instagram',
       'Twitter'], dtype=object)

In [7]:
df_input['Country'].unique()

array(['China', 'Russia'], dtype=object)

### 2. NLP text processing

In [8]:
!pip install emoji

In [9]:
#FINAL SBERT FRIENDLY LIGHT CLEANING
#(Aggressive cleaning destroy the semantic richness of the sentences)

import os, re
import pandas as pd
from tqdm import tqdm
from collections import Counter
import emoji


# -------------------------------------------------
# LIGHT CLEANING FUNCTION (FOR SBERT INPUT)
# -------------------------------------------------
def clean_text_light(text):

    if not isinstance(text, str):
        return ""

    # 1. Convert emoji to text FIRST
    try:
        text = emoji.demojize(text)
    except Exception:
        pass

    # 2. Now convert to lowercase
    text = text.lower()

    # if not isinstance(text, str):
    #     return ""

    # text = text.lower()

    # # Convert emoji to text (optional but useful)
    # try:
    #     text = emoji.demojize(text)
    # except Exception:
    #     pass

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove mentions AND the "rt" marker
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"\brt\b", " ", text)

    # Remove hashtag symbol but keep the word
    text = text.replace("#", "")

    # Remove weird characters but KEEP numbers and sentence structure
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\:]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# -------------------------------------------------
# CSV PROCESSING
# -------------------------------------------------
def process_csv_file(infile, outfile=None, chunk_size=20000, update_counter=None):

    if outfile is None:
        outfile = infile.replace(".csv", "_clean.csv")

    os.makedirs(os.path.dirname(outfile) or ".", exist_ok=True)

    reader = pd.read_csv(infile, chunksize=chunk_size, encoding='utf-8',
                         dtype=str, on_bad_lines='skip')

    first_chunk = True

    for chunk in tqdm(reader, desc=os.path.basename(infile)):
        chunk = chunk.fillna("")

        if "Description" not in chunk.columns:
            raise ValueError("Column 'Description' not found in CSV!")

        chunk['clean_text'] = chunk['Description'].apply(clean_text_light)

        # Counter for visualization only (optional)
        if update_counter is not None:
            for text in chunk['clean_text']:
                if text:
                    update_counter.update(text.split())

        if first_chunk:
            chunk.to_csv(outfile, index=False, mode='w', encoding='utf-8')
            first_chunk = False
        else:
            chunk.to_csv(outfile, index=False, mode='a', header=False, encoding='utf-8')

    return outfile

# -------------------------------------------------
# RUN PIPELINE
# -------------------------------------------------
def run_pipeline(input_file, output_folder, chunk_size=2000):

    os.makedirs(output_folder, exist_ok=True)
    total_counter = Counter()

    base_name = os.path.basename(input_file).replace(".csv", "")
    outfile = os.path.join(output_folder, f"{base_name}_clean.csv")

    process_csv_file(
        infile=input_file,
        outfile=outfile,
        chunk_size=chunk_size,
        update_counter=total_counter
    )

    print(f"\n✔ Processed: {input_file}")
    print(f"➡ Clean saved: {outfile}")

    return total_counter

In [10]:
if __name__ == "__main__":
    import os
    import pandas as pd

    # Load your data
    #df = pd.read_csv("/Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/8_light_cleaning_with clustering/sample_100000.csv")

    #OUTPUT_FOLDER = "/content/drive/MyDrive/FYR_UOK/Final yr research UOK/OBJ-1/Global_data_process/Subsets/A40_gold_outputs"
    OUTPUT_FOLDER = "/Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/14_final_hybrid approach/A40_OUTPUTS"
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Save df to a CSV for pipeline
    temp_path = os.path.join(OUTPUT_FOLDER, "gold.csv")
    df_input.to_csv(temp_path, index=False, encoding="utf-8")

    # Run SBERT-friendly pipeline
    run_pipeline(temp_path, OUTPUT_FOLDER, chunk_size=2000)

gold.csv: 213it [02:30,  1.41it/s]


✔ Processed: /Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/14_final_hybrid approach/A40_OUTPUTS/gold.csv
➡ Clean saved: /Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/14_final_hybrid approach/A40_OUTPUTS/gold_clean.csv


#### 2.1 Readying for the clustering

In [11]:
#df_cleaned = pd.read_csv("/content/drive/MyDrive/FYR_UOK/Final yr research UOK/OBJ-1/Global_data_process/Subsets/A40_gold_outputs/gold_clean.csv")
df_cleaned = pd.read_csv("/Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/14_final_hybrid approach/A40_OUTPUTS/gold_clean.csv")

In [12]:
df_cleaned.head()

,Platform,Country,Description,clean_text
0,TikTok,China,Brave panda conquers raging river in Shaanxi! ...,brave panda conquers raging river in shaanxi! ...
1,TikTok,China,Israeli Air Force planes launched an airstrike...,israeli air force planes launched an airstrike...
2,TikTok,China,The far-right Alternative for Germany (AfD) pa...,the far right alternative for germany afd part...
3,TikTok,China,Chechen leader Ramzan Kadyrov shared a video s...,chechen leader ramzan kadyrov shared a video s...
4,TikTok,China,An explosion at a Russian military depot in th...,an explosion at a russian military depot in th...


In [13]:
df_cleaned.shape

(425000, 4)

In [14]:
df_cleaned.isnull().sum()

Platform          0
Country           0
Description     624
clean_text     1785
dtype: int64

In [15]:
type(df_cleaned['clean_text'])

pandas.core.series.Series

In [16]:
# 1. Fill empty (NaN) values with a blank string so they are not 'floats'
df_cleaned['clean_text'] = df_cleaned['clean_text'].fillna("")

# 2. Force the entire column to be strings (just in case)
df_cleaned['clean_text'] = df_cleaned['clean_text'].astype(str)

docs = df_cleaned['clean_text'].tolist() #SBERT expects (Pandas series → Python list)

In [17]:
type(docs)

list

In [18]:
df_cleaned.isnull().sum()

Platform         0
Country          0
Description    624
clean_text       0
dtype: int64

In [19]:
df_cleaned['clean_text'].isnull().sum()

0

### 3. Gold data clustering with BERTopic

In [20]:
!pip install bertopic gensim

  Obtaining dependency information for FuzzyTM>=0.4.0 from https://files.pythonhosted.org/packages/2d/30/074bac7a25866a2807c1005c7852c0139ac22ba837871fc01f16df29b9dc/FuzzyTM-2.0.9-py3-none-any.whl.metadata
  Obtaining dependency information for pyfume from https://files.pythonhosted.org/packages/ed/ea/a3b120e251145dcdb10777f2bc5f18b1496fd999d705a178c1b0ad947ce1/pyFUME-0.3.4-py3-none-any.whl.metadata
  Obtaining dependency information for scipy>=1.7.0 from https://files.pythonhosted.org/packages/0d/3e/d05b9de83677195886fb79844fcca19609a538db63b1790fa373155bc3cf/scipy-1.10.1-cp311-cp311-macosx_12_0_arm64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 kB 1.2 MB/s eta 0:00:00 MB/s eta 0:00:01
  Obtaining dependency information for simpful==2.12.0 from https://files.pythonhosted.org/packages/9d/0e/aebc2fb0b0f481994179b2ee2b8e6bbf0894d971594688c018375e7076ea/simpful-2.12.0-py3-none-any.whl.metadata
  Preparing metadata (setup.py) ... done
  Obtaining dependency informa

In [21]:
import os
import warnings
import numpy as np
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

In [22]:
# Suppress minor warnings to keep the terminal clean
warnings.filterwarnings("ignore")

# =============================================================================
# 1. SETUP PATHS
# =============================================================================
#SAVE_DIR = "/content/drive/MyDrive/FYR_UOK/Final yr research UOK/OBJ-1/Global_data_process/Subsets/A40_gold_outputs"
SAVE_DIR = "/Users/subhashbandaraekanayake/Desktop/4.1/Final yr research UOK/3_Git_clone/UOK_FYR_2/14_final_hybrid approach/A40_OUTPUTS"
os.makedirs(SAVE_DIR, exist_ok=True)

SBERT_MODEL_PATH = os.path.join(SAVE_DIR, "sbert_embedding_model")
BERTOPIC_MODEL_PATH = os.path.join(SAVE_DIR, "gold_bertopic_best_model")
EMBEDDINGS_PATH = os.path.join(SAVE_DIR, "gold_embeddings.npy")
RESULTS_CSV_PATH = os.path.join(SAVE_DIR, "gold_topic_results.csv")
METRICS_CSV_PATH = os.path.join(SAVE_DIR, "model_metrics.csv")

In [23]:
# =============================================================================
# 2. HELPER FUNCTIONS FOR SCORING (Raw Scores)
# =============================================================================
def calculate_coherence(topic_model, docs):
    tokenizer = vectorizer_model.build_analyzer()
    tokens = [tokenizer(doc) for doc in docs]
    dictionary = corpora.Dictionary(tokens)

    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in words]
        for topic_id, words in topics.items()
        if topic_id != -1 and words
    ]

    if len(topic_words) == 0:
        return 0.0

    coherence_model = CoherenceModel(
        topics=topic_words, texts=tokens, dictionary=dictionary, coherence='c_v'
    )
    return coherence_model.get_coherence()

def calculate_outlier_ratio(topics):
    total = len(topics)
    outliers = sum(1 for t in topics if t == -1)
    return outliers / total if total > 0 else 1.0

In [24]:
# =============================================================================
# 3. GENERATE EMBEDDINGS
# =============================================================================
print("Initializing SBERT Model (all-mpnet-base-v2)...")
embedding_model = SentenceTransformer("all-mpnet-base-v2")

print("Generating Embeddings... (This may take a while)")
# Note: Ensure 'docs' is defined as your list of cleaned texts before running this script
embeddings = embedding_model.encode(
    docs,
    show_progress_bar=True,
    convert_to_numpy=True,       # BERTopic expects numpy
    normalize_embeddings=True)

vectorizer_model = CountVectorizer(stop_words="english")

# Save the SentenceTransformer embedding model
embedding_model.save(SBERT_MODEL_PATH)

Initializing SBERT Model (all-mpnet-base-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating Embeddings... (This may take a while)


Batches:   0%|          | 0/13282 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
# =============================================================================
# 4. INITIALIZE & TRAIN THE MODEL (Fixed Parameters)
# =============================================================================
print("\n Building BERTopic with fixed parameters...")
print("   UMAP: n_neighbors=15, n_components=5")
print("   HDBSCAN: min_cluster_size=500, min_samples=15")

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=500,
    min_samples=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
    gen_min_span_tree=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True
)

print("----> Training BERTopic Model...")
topics, probs = topic_model.fit_transform(docs, embeddings)


 Building BERTopic with fixed parameters...
   UMAP: n_neighbors=15, n_components=5
   HDBSCAN: min_cluster_size=500, min_samples=15


NameError: name 'UMAP' is not defined

In [ ]:
# =============================================================================
# 5. CALCULATE RAW VALIDATION SCORES
# =============================================================================
print("\nCalculating Validation Scores...")

num_topics = len(set(topics)) - (1 if -1 in topics else 0)
outlier_ratio = calculate_outlier_ratio(topics)

# 1. DBCV (built into HDBSCAN)
try:
    dbcv = float(hdbscan_model.relative_validity_)
except AttributeError:
    dbcv = 0.0

# 2. Silhouette (only on non-noise documents)
valid_indices = [i for i, t in enumerate(topics) if t != -1]
if len(set(topics[i] for i in valid_indices)) > 1 and len(valid_indices) > 0:
    valid_embeddings = embeddings[valid_indices]
    valid_topics = [topics[i] for i in valid_indices]
    silhouette = silhouette_score(valid_embeddings, valid_topics, metric='cosine')
else:
    silhouette = -1.0

# 3. Topic Coherence (c_v)
coherence = calculate_coherence(topic_model, docs)

print(f"✅ Total Topics: {num_topics}")
print(f"✅ Outlier Ratio: {outlier_ratio:.2%}")
print(f"✅ DBCV Score: {dbcv:.4f}")
print(f"✅ Silhouette Score: {silhouette:.4f}")
print(f"✅ Coherence Score: {coherence:.4f}")

In [ ]:
# =============================================================================
# 6. SAVE ALL OUTPUTS
# =============================================================================
print("\n Saving models, embeddings, and results...")

# Save the mathematical vectors
np.save(EMBEDDINGS_PATH, embeddings)

# Save the SentenceTransformer embedding model
embedding_model.save(SBERT_MODEL_PATH)

# Save the BERTopic model
topic_model.save(BERTOPIC_MODEL_PATH)

# Save document-topic results
# Extract the probability for each doc's assigned topic
if probs is not None and hasattr(probs, 'ndim') and probs.ndim == 2:
    assigned_probs = probs[np.arange(len(topics)), topics]
else:
    assigned_probs = probs if probs is not None else [0.0] * len(topics)

df_results = pd.DataFrame({
    "document": docs,
    "topic": topics,
    "assigned_probability": assigned_probs,
})
df_results.to_csv(RESULTS_CSV_PATH, index=False)

# Save the raw metrics to a CSV for your records
df_metrics = pd.DataFrame([{
    "n_neighbors": 15,
    "n_components": 5,
    "min_cluster_size": 500,
    "min_samples": 15,
    "num_topics": num_topics,
    "outlier_ratio": outlier_ratio,
    "dbcv_raw": dbcv,
    "silhouette_raw": silhouette,
    "coherence_raw": coherence
}])
df_metrics.to_csv(METRICS_CSV_PATH, index=False)

print("\n All processes complete! Everything is securely saved in your Google Drive.")